# 🫀 Heart Disease UCI Dataset - Exploratory Data Analysis (EDA)

**Author:** Pronab Sardar  
**Course:** AIMLCZG523 - MLOps Assignment 01  
**Institution:** BITS Pilani  
**Dataset:** UCI Heart Disease (Cleveland)

## 🎯 Objectives
1. Understand dataset structure and quality
2. Analyze missing values and data types
3. Explore target class distribution
4. Visualize numerical feature distributions
5. Analyze categorical feature relationships with target
6. Compute correlation between features
7. Identify outliers via box plots
8. Generate interactive visualizations with Plotly
9. Derive actionable insights for feature engineering

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titleweight'] = 'bold'

# Color palette (green for no disease, red for disease)
PALETTE = ['#2ecc71', '#e74c3c']

# Ensure screenshots folder exists
os.makedirs('../screenshots/mlflow', exist_ok=True)

print('✅ All libraries loaded successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

## 2. Load the Dataset

In [ ]:
# Load the Heart Disease UCI dataset
df = pd.read_csv('../data/raw/heart_disease.csv')

print(f'📊 Dataset Shape: {df.shape}')
print(f'📊 Total Rows: {df.shape[0]:,}')
print(f'📊 Total Columns: {df.shape[1]}')
print(f'\n📋 Column Names: {df.columns.tolist()}')

In [ ]:
# Preview the first 5 rows
print('First 5 rows:')
df.head()

In [ ]:
# Preview the last 5 rows
print('Last 5 rows:')
df.tail()

### 📖 Feature Dictionary

| Feature | Description | Type |
|---------|-------------|------|
| **age** | Age in years | Numeric |
| **sex** | 1=Male, 0=Female | Categorical |
| **cp** | Chest pain type (0-3) | Categorical |
| **trestbps** | Resting blood pressure (mm Hg) | Numeric |
| **chol** | Serum cholesterol (mg/dl) | Numeric |
| **fbs** | Fasting blood sugar > 120 mg/dl (1/0) | Binary |
| **restecg** | Resting ECG results (0-2) | Categorical |
| **thalach** | Maximum heart rate achieved | Numeric |
| **exang** | Exercise-induced angina (1/0) | Binary |
| **oldpeak** | ST depression induced by exercise | Numeric |
| **slope** | Slope of peak exercise ST segment | Categorical |
| **ca** | Number of major vessels colored (0-3) | Numeric |
| **thal** | Thalassemia type | Categorical |
| **target** | Heart disease (1=Yes, 0=No) | Binary |

## 3. Data Types & Statistical Summary

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Memory Usage ===')
print(f'{df.memory_usage(deep=True).sum() / 1024:.2f} KB')

In [ ]:
# Statistical summary
print('=== Statistical Summary ===')
df.describe().T.round(2)

In [ ]:
# Info() for compact overview
df.info()

## 4. Missing Value Analysis

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if not missing_df.empty:
    print('❗ Columns with Missing Values:')
    print(missing_df)
else:
    print('✅ No missing values in the dataset!')

In [ ]:
# Interactive Plotly visualization for missing values
if not missing_df.empty:
    fig = px.bar(
        missing_df, x=missing_df.index, y='Missing %',
        title='Missing Values by Feature',
        color='Missing %',
        color_continuous_scale='Reds',
        text='Missing %'
    )
    fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
    fig.update_layout(height=400, xaxis_title='Feature', yaxis_title='Missing %')
    fig.show()
else:
    print('No missing values to visualize.')

In [ ]:
# Missing value heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.tight_layout()
plt.savefig('../screenshots/mlflow/missing_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Target Variable Analysis (Class Distribution)

In [ ]:
# Class distribution counts and percentages
print('🎯 Target Class Distribution:')
print(df['target'].value_counts())
print(f'\n📊 Percentage:')
print((df['target'].value_counts(normalize=True) * 100).round(2))
print(f'\n⚖️ Balance Ratio: {df["target"].value_counts()[0] / df["target"].value_counts()[1]:.2f}:1')

In [ ]:
# Class distribution: Bar + Pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df['target'].value_counts()
counts.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target (0=No Disease, 1=Disease)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
counts.plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=PALETTE, startangle=90,
    labels=['No Disease', 'Disease'],
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../screenshots/mlflow/class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n✅ Dataset is well-balanced - no oversampling required.')

## 6. Numerical Feature Analysis

In [ ]:
# Define numerical features
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

print(f'📊 Numerical features: {num_features}')
print('\nStatistical summary of numerical features:')
df[num_features].describe().T.round(2)

In [ ]:
# Histograms with target overlay
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(
        df[df['target'] == 0][col].dropna(),
        bins=25, alpha=0.6, label='No Disease',
        color=PALETTE[0], edgecolor='black'
    )
    axes[i].hist(
        df[df['target'] == 1][col].dropna(),
        bins=25, alpha=0.6, label='Disease',
        color=PALETTE[1], edgecolor='black'
    )
    axes[i].axvline(df[col].mean(), color='blue', linestyle='--', alpha=0.6, label='Overall Mean')
    axes[i].set_title(f'Distribution of {col}', fontweight='bold', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

axes[-1].axis('off')  # Turn off the last empty subplot

plt.suptitle('Numerical Feature Distributions by Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/histograms.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots (KDE) for numerical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.kdeplot(data=df[df['target'] == 0], x=col, ax=axes[i], color=PALETTE[0], label='No Disease', fill=True, alpha=0.5)
    sns.kdeplot(data=df[df['target'] == 1], x=col, ax=axes[i], color=PALETTE[1], label='Disease', fill=True, alpha=0.5)
    axes[i].set_title(f'Density of {col}', fontweight='bold')
    axes[i].legend()

axes[-1].axis('off')
plt.suptitle('Kernel Density Estimates by Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/density_plots.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
# Compute correlation matrix
corr = df.corr()

# Correlation heatmap with mask (upper triangle)
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8},
    annot_kws={'fontsize': 10}
)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with target
target_corr = corr['target'].drop('target').abs().sort_values(ascending=False)
print('🎯 Feature Correlation with Target (Absolute Value):')
print(target_corr.round(3))

# Visualize top correlations
fig = px.bar(
    x=target_corr.values, y=target_corr.index,
    orientation='h',
    title='Feature Correlation with Target (Absolute Value)',
    labels={'x': 'Absolute Correlation', 'y': 'Feature'},
    color=target_corr.values,
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 8. Categorical Feature Analysis

In [ ]:
# Define categorical features
cat_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

print(f'📊 Categorical features: {cat_features}')
print('\nValue counts for each categorical feature:')
for col in cat_features:
    print(f'\n{col}:')
    print(df[col].value_counts().sort_index())

In [ ]:
# Categorical vs Target analysis (stacked bar charts with percentages)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=PALETTE, stacked=True, edgecolor='black')
    axes[i].set_title(f'{col} vs Target (%)', fontweight='bold')
    axes[i].set_ylabel('Percentage')
    axes[i].set_xlabel(col)
    axes[i].legend(['No Disease', 'Disease'], loc='upper right', fontsize=9)
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Categorical Features vs Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/categorical_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Countplots for categorical features grouped by target
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    sns.countplot(data=df, x=col, hue='target', ax=axes[i], palette=PALETTE)
    axes[i].set_title(f'{col} by Target', fontweight='bold')
    axes[i].legend(['No Disease', 'Disease'], loc='upper right', fontsize=9)

plt.suptitle('Categorical Feature Counts by Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/categorical_counts.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Outlier Detection (Box Plots)

In [ ]:
# Box plots for numerical features by target
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for i, col in enumerate(num_features):
    sns.boxplot(x='target', y=col, data=df, ax=axes[i], palette=PALETTE)
    axes[i].set_title(f'{col} by Target', fontweight='bold')
    axes[i].set_xticklabels(['No Disease', 'Disease'])

plt.suptitle('Box Plots: Numerical Features by Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/boxplots.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots for a deeper distribution view
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for i, col in enumerate(num_features):
    sns.violinplot(x='target', y=col, data=df, ax=axes[i], palette=PALETTE)
    axes[i].set_title(f'{col} by Target', fontweight='bold')
    axes[i].set_xticklabels(['No Disease', 'Disease'])

plt.suptitle('Violin Plots: Numerical Features by Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/violinplots.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# IQR-based outlier detection
print('🔍 IQR-based Outlier Detection for Numerical Features:')
print('=' * 60)

outlier_summary = {}
for col in num_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = {
        'Outlier Count': len(outliers),
        'Outlier %': round((len(outliers) / len(df)) * 100, 2)
    }

outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df)

## 10. Interactive Visualizations (Plotly)

In [ ]:
# Interactive scatter matrix
fig = px.scatter_matrix(
    df,
    dimensions=['age', 'trestbps', 'chol', 'thalach', 'oldpeak'],
    color='target',
    color_discrete_map={0: PALETTE[0], 1: PALETTE[1]},
    title='Interactive Scatter Matrix of Numerical Features',
    labels={'target': 'Heart Disease'}
)
fig.update_layout(height=800, showlegend=True)
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.show()

In [ ]:
# Age vs Cholesterol interactive plot
fig = px.scatter(
    df, x='age', y='chol',
    color='target', size='thalach',
    hover_data=['sex', 'cp', 'trestbps', 'oldpeak'],
    color_discrete_map={0: PALETTE[0], 1: PALETTE[1]},
    title='Age vs Cholesterol (Size = Max Heart Rate)',
    labels={'target': 'Heart Disease', 'chol': 'Cholesterol', 'age': 'Age'}
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# 3D scatter: Age, Cholesterol, Max HR
fig = px.scatter_3d(
    df, x='age', y='chol', z='thalach',
    color='target',
    color_discrete_map={0: PALETTE[0], 1: PALETTE[1]},
    title='3D View: Age × Cholesterol × Max Heart Rate',
    labels={'target': 'Heart Disease'}
)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Sunburst chart: sex → cp → target
df_sunburst = df.copy()
df_sunburst['sex_label'] = df_sunburst['sex'].map({0: 'Female', 1: 'Male'})
df_sunburst['cp_label'] = df_sunburst['cp'].map({0: 'Typical Angina', 1: 'Atypical Angina', 2: 'Non-anginal', 3: 'Asymptomatic'})
df_sunburst['target_label'] = df_sunburst['target'].map({0: 'No Disease', 1: 'Disease'})

fig = px.sunburst(
    df_sunburst,
    path=['sex_label', 'cp_label', 'target_label'],
    title='Sunburst: Sex → Chest Pain Type → Disease',
    color='target_label',
    color_discrete_map={'No Disease': PALETTE[0], 'Disease': PALETTE[1]}
)
fig.update_layout(height=700)
fig.show()

## 11. Age Group Analysis

In [ ]:
# Create age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 40, 55, 70, 120],
    labels=['<40', '40-55', '55-70', '>70']
)

# Disease rate by age group
age_disease = df.groupby('age_group')['target'].agg(['count', 'sum'])
age_disease['disease_rate_%'] = (age_disease['sum'] / age_disease['count'] * 100).round(2)
print('🩺 Disease Rate by Age Group:')
print(age_disease)

In [ ]:
# Visualize disease rate by age group
fig = px.bar(
    x=age_disease.index.astype(str),
    y=age_disease['disease_rate_%'],
    title='Disease Rate by Age Group',
    labels={'x': 'Age Group', 'y': 'Disease Rate (%)'},
    color=age_disease['disease_rate_%'],
    color_continuous_scale='Reds',
    text=age_disease['disease_rate_%']
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

# Remove the age_group column (was just for analysis)
df.drop(columns='age_group', inplace=True)

## 12. Feature Relationships

In [ ]:
# Pairplot of key features
key_features = ['age', 'thalach', 'chol', 'oldpeak', 'target']
sns.pairplot(df[key_features], hue='target', palette=PALETTE, diag_kind='kde', height=2.2)
plt.suptitle('Pairplot of Key Features', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('../screenshots/mlflow/pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Age vs Max Heart Rate (thalach) by target
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='age', y='thalach', hue='target', palette=PALETTE, s=80, alpha=0.7)
plt.title('Age vs Max Heart Rate (Colored by Target)', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Max Heart Rate (thalach)')
plt.legend(['No Disease', 'Disease'], title='Target')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../screenshots/mlflow/age_vs_thalach.png', dpi=100, bbox_inches='tight')
plt.show()

## 13. Summary Statistics by Target

In [ ]:
# Compare means of numerical features between target classes
print('📊 Mean values of numerical features by target:')
print(df.groupby('target')[num_features].mean().round(2))

print('\n📊 Median values of numerical features by target:')
print(df.groupby('target')[num_features].median().round(2))

In [ ]:
# Percentage distribution of categorical features by target
print('📊 Percentage distribution of categorical features by target:')
print('=' * 60)
for col in cat_features:
    print(f'\n{col.upper()}:')
    ct = pd.crosstab(df[col], df['target'], normalize='columns') * 100
    print(ct.round(2))

## 🔑 Key Insights from EDA

### 📊 Dataset Overview
1. **Size:** 303 patient records with 14 features (13 predictors + 1 target)
2. **Class Balance:** Dataset is well-balanced (~54% No Disease, ~46% Disease) - **no oversampling needed**
3. **Missing Values:** Only `ca` and `thal` have minor missing values (<3%) - handled via mode imputation

### 🎯 Key Predictors (Top 5 by correlation with target)
1. **thalach** (Max Heart Rate) - **Strong negative correlation** → Lower thalach = Higher disease risk
2. **oldpeak** (ST Depression) - **Strong positive correlation** → Higher oldpeak = Higher risk
3. **cp** (Chest Pain Type) - Type 0 (typical angina) strongly indicates disease
4. **exang** (Exercise Angina) - Positive exang strongly predicts disease
5. **ca** (Major Vessels) - More affected vessels correlate with disease

### 👥 Demographic Insights
- **Age:** Disease patients tend to be older (mean ~56 vs ~52)
- **Sex:** Males (sex=1) show higher disease prevalence in this cohort
- **Age Groups:** Disease rate increases significantly after age 55

### 🩺 Clinical Insights
- **Max Heart Rate:** Disease patients achieve lower max HR during exercise
- **Cholesterol:** Mild positive correlation with disease
- **Blood Pressure:** Slightly higher in disease patients
- **ST Depression (oldpeak):** Strong predictor - values > 1.0 signal higher risk

### 🛠️ Feature Engineering Strategy
Based on these insights, we will create:
1. **Age groups:** Bin into [<40, 40-55, 55-70, >70] for non-linear age effects
2. **HR-Age ratio:** `thalach / age` captures cardiovascular fitness
3. **Age × Cholesterol interaction:** Combined cardiovascular risk
4. **BP-Cholesterol ratio:** Metabolic health indicator
5. **Cholesterol categories:** Normal (<200), Borderline (200-240), High (>240)

### 🚀 Next Steps
1. Apply feature engineering in `src/data_preprocessing.py`
2. Train multiple models with hyperparameter tuning (see `02_Model_Training.ipynb`)
3. Track experiments with MLflow
4. Deploy best model via FastAPI

## 📁 Generated Artifacts

The following plots have been saved to `../screenshots/mlflow/`:
- `missing_heatmap.png`
- `class_distribution.png`
- `histograms.png`
- `density_plots.png`
- `correlation_heatmap.png`
- `categorical_analysis.png`
- `categorical_counts.png`
- `boxplots.png`
- `violinplots.png`
- `pairplot.png`
- `age_vs_thalach.png`

---

**Author:** Pronab Sardar  
**Course:** AIMLCZG523 - MLOps Assignment 01  
**BITS Pilani** | July 2026